# 使用 Selenium 抓取客户端渲染页面

## 练习目标（理念）

很多现代网站靠 **JavaScript** 在浏览器里渲染内容；只用 `requests` 拿到的往往是空壳 HTML。  
本练习用 **Selenium** 启动真实浏览器，等页面渲染后再取 `page_source`，再交给 **OpenAI** 做俏皮摘要——对应第 1 周 Day 1「抓网页 + LLM 摘要」，只是抓取层换成浏览器自动化。

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 网页抓取 | Selenium `webdriver.Chrome` + `page_source` |
| Chat Completions | `openai.chat.completions.create(...)` |
| `messages`（system / user） | 俏皮摘要的 system / user prompt |
| 环境变量 | `.env` 里的 `OPENAI_API_KEY` |

## 怎么跑

1. 安装依赖：`pip install selenium`（并确保本机有 Chrome / ChromeDriver）
2. 准备好 `.env`：至少有 `OPENAI_API_KEY`
3. 从上到下依次运行单元格；先测抓取，再跑摘要


## 步骤 1：用 Selenium 读取「渲染后」的 HTML

下面定义 `readClientRenderedHTML(url)`：打开 Chrome → 访问 URL → 短暂等待 JS 渲染 → 返回完整 HTML 字符串。


In [ ]:
# ========== 导入：Selenium 浏览器自动化相关 ==========

# 从 selenium 导入 webdriver：用来启动/控制浏览器（这里用 Chrome）
from selenium import webdriver
# Options：配置 Chrome 启动参数（例如是否无头 headless）
from selenium.webdriver.chrome.options import Options
# By：按 CSS/ID 等定位元素（本格虽导入，函数体未用到；保留原导入）
from selenium.webdriver.common.by import By
# time：用 sleep 做简单「等页面渲染」——粗暴但够演示
import time

def readClientRenderedHTML(url):
    """用真实 Chrome 打开 url，等待前端渲染后返回 page_source（完整 HTML）。"""
    # 创建 Chrome 启动选项对象
    options = Options()
    # 若取消下一行注释，则无界面（headless）运行；调试时建议先开可见窗口
    #options.add_argument("--headless")  # 无界面运行浏览器

    # 按 options 启动 Chrome；需本机已装浏览器与匹配的驱动
    driver = webdriver.Chrome(options=options)

    # 导航到目标 URL（此时浏览器开始加载并执行页面 JS）
    driver.get(url)

    # 固定睡 3 秒：给客户端渲染留时间（生产环境更常用显式等待 WebDriverWait）
    time.sleep(3)

    # page_source：当前 DOM 序列化后的 HTML（含 JS 已写入的内容）
    html = driver.page_source


    # 关闭浏览器进程，释放资源（务必调用，避免残留 chromedriver）
    driver.quit()

    # 把渲染后的 HTML 字符串交还给调用方
    return html


## 步骤 2：测试抓取函数

直接运行下一格：对个人站点调用 `readClientRenderedHTML`，并用 `print` 查看返回的 HTML。


In [ ]:
# ========== 测试：确认 Selenium 真的拿到了渲染后的页面 ==========

# 调用上面定义的函数，抓取个人站点（URL 字符串保持原样）
result  = readClientRenderedHTML("https://khaldoon-saqallah.com")
# 打印完整 HTML，便于肉眼检查是否含正文（而不只是空壳）
print(result)


## 步骤 3：抓取 + AI 摘要（对齐 Day 1）

下面把 Selenium 抓到的页面文本，塞进 Chat Completions 的 `messages`，让模型生成**俏皮、幽默**的 Markdown 摘要。


In [ ]:
# ========== Selenium 抓页 + OpenAI 俏皮摘要 ==========

# 导入标准库 os：从环境变量读取 API Key
import os
# load_dotenv：把 .env 中的密钥读进进程环境，避免把密钥写进笔记本
from dotenv import load_dotenv
# Markdown + display：在 Jupyter 里漂亮渲染模型返回的 Markdown
from IPython.display import Markdown, display
# OpenAI 客户端：调用云端 Chat Completions API
from openai import OpenAI

# 加载 .env；override=True 表示已存在的环境变量也用 .env 覆盖
load_dotenv(override=True)
# 读取 OPENAI_API_KEY（名字必须与 .env 里一致）
api_key = os.getenv('OPENAI_API_KEY')

# 用该密钥创建 OpenAI 客户端实例
openai = OpenAI(api_key=api_key)

# system prompt：规定「俏皮助手」人设与输出格式（英文指令保留，改译会改变模型行为）
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

# user prompt 前缀：告诉模型要做什么（后面会拼接网页正文）
user_prompt = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

def messages_for(website):
    """把 system + user（含网页正文）组装成 Chat Completions 需要的 messages 列表。"""
    return [
        # system：定角色与风格
        {"role": "system", "content": system_prompt},
        # user：任务说明 + 网页 HTML/文本
        {"role": "user", "content": user_prompt + "\n\n" + website}
    ]


def summarize(url):
    """抓取 url 的渲染 HTML，再调用 gpt-4.1-mini 生成摘要文本。"""
    # 先用 Selenium 拿到客户端渲染后的页面内容
    website = readClientRenderedHTML(url)
    # 调用 Chat Completions：model / messages 参数保持原样
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    # 从响应里取出第一条 choice 的助手回复正文
    return response.choices[0].message.content

def display_summary(url):
    """summarize 后在笔记本中以 Markdown 显示。"""
    # 拿到模型摘要字符串
    summary = summarize(url)
    # 在输出区渲染为 Markdown
    display(Markdown(summary))

 


## 步骤 4：调用 `display_summary`

对目标 URL 跑一遍完整流水线：Selenium 抓取 → OpenAI 摘要 → 笔记本展示。


In [ ]:
# ========== 端到端：对目标站点展示俏皮摘要 ==========

# 调用上一格定义的 display_summary（URL 保持原样）
display_summary("https://khaldoon-saqallah.com")
